# Batch layer trên Google Colab: CSE-CIC-IDS2018 với PySpark

Notebook chạy toàn bộ batch layer trên Colab: ETL và báo cáo chất lượng dữ liệu, kịch bản 1, RF đa lớp, kịch bản 2, benchmark khả năng mở rộng, biểu đồ.

| Thành phần | Vị trí | Ghi chú |
|---|---|---|
| Mã nguồn | `MyDrive/ie212-ids2018` | Clone từ GitHub, tự `git pull` ở mỗi phiên |
| Dữ liệu CSV | `MyDrive/ie212_data` nếu đã có đủ 10 file, nếu không thì `/content/data` | Tự tải từ AWS Open Data khi thiếu; `/content` mất khi ngắt phiên |
| Parquet trung gian | `/content/work/parquet`, sao lưu ở `MyDrive/ie212_parquet` (~3,4 GB) | Phiên sau khôi phục từ Drive, không phải chạy lại ETL |
| Kết quả, mô hình | `MyDrive/ie212-ids2018/colab_out/{results,models}` | Tách riêng khỏi `results/` và `models/` của repo |

Chọn runtime **CPU + High-RAM** (Colab Pro: 8 vCPU, 50 GB RAM); GPU không làm Spark nhanh hơn. Ô thiết lập tự tính bộ nhớ driver, số phân vùng shuffle và số core benchmark theo máy. Trên Colab Pro High-RAM, toàn bộ notebook chạy khoảng 10 giờ.

Mọi bước đều chạy tiếp được: kết quả nằm trên Drive, kịch bản 2 và benchmark có `--resume`. Khi bị ngắt phiên, chạy lại ba ô **Thiết lập** rồi chạy tiếp từ bước đang dở.

## 0. Thiết lập (chạy lại ở mỗi phiên)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, psutil

REPO_URL       = 'https://github.com/phonguytin/ie212-ids2018.git'
PROJECT        = '/content/drive/MyDrive/ie212-ids2018'
RESULTS        = f'{PROJECT}/colab_out/results'
MODELS         = f'{PROJECT}/colab_out/models'
DATA_DRIVE     = '/content/drive/MyDrive/ie212_data'
PARQUET_BACKUP = '/content/drive/MyDrive/ie212_parquet'
WORK           = '/content/work'
DATA_LOCAL     = '/content/data'

if not os.path.exists(f'{PROJECT}/.git'):
    !git clone -q {REPO_URL} "{PROJECT}"
else:
    !git -C "{PROJECT}" pull -q --ff-only
!git -C "{PROJECT}" log --oneline -1

# Cấu hình "Cân bằng" cho Colab Pro High-RAM. Colab miễn phí (2 vCPU, ~12 GB): S1_TRAIN_FRAC = 0.3, S2_TRAIN_FRAC = 0.2
N_CPU  = os.cpu_count()
RAM_GB = int(psutil.virtual_memory().total / 2**30)

S1_TRAIN_FRAC = 1.0
S2_TRAIN_FRAC = 0.5
S2_PRESET     = 'fast'
BENCH_CORES   = ','.join(str(c) for c in (1, 2, 4, 8, 16) if c <= N_CPU)
BENCH_FRACS   = '0.25,0.5,1.0'
BENCH_TIMEOUT = 3600
DRIVER_MEM    = f'{int(RAM_GB * 0.65)}g'
BENCH_MEM     = f'{int(RAM_GB * 0.5)}g'
SHUFFLE_PARTS = N_CPU * 8
print(f'{N_CPU} vCPU, {RAM_GB} GB RAM | driver {DRIVER_MEM} | bench {BENCH_CORES} cores, {BENCH_MEM} | shuffle {SHUFFLE_PARTS}')

assert os.path.exists(f'{PROJECT}/src/ids/etl.py'), f'src/ids not found in {PROJECT}: {os.listdir(PROJECT) if os.path.isdir(PROJECT) else "missing"}'
for d in (RESULTS, MODELS, WORK, DATA_LOCAL):
    os.makedirs(d, exist_ok=True)
os.chdir(PROJECT)

os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['PYTHONPATH'] = f'{PROJECT}/src'
os.environ['SPARK_DRIVER_MEMORY'] = DRIVER_MEM
os.environ['SPARK_SHUFFLE_PARTITIONS'] = str(SHUFFLE_PARTS)
if f'{PROJECT}/src' not in sys.path:
    sys.path.insert(0, f'{PROJECT}/src')
print(os.listdir(PROJECT))

In [ ]:
!test -x /usr/lib/jvm/java-17-openjdk-amd64/bin/java || (apt-get -qq update && apt-get -qq install -y openjdk-17-jdk-headless > /dev/null)
!pip -q install pyspark==4.0.1 awscli

!java -version 2>&1 | grep -i version
!python -c "import pyspark, ids; print('pyspark', pyspark.__version__, '| ids', ids.__file__)"
!nproc; free -g | head -2; df -h /content | tail -1

In [ ]:
def n_csv(d):
    return len([f for f in os.listdir(d) if f.endswith('.csv')]) if os.path.isdir(d) else 0

def data_dir():
    for d in (DATA_DRIVE, f'{PROJECT}/data'):
        if n_csv(d) >= 10:
            return d
    if n_csv(DATA_LOCAL) < 10:
        !aws s3 cp --no-sign-request --only-show-errors "s3://cse-cic-ids2018/Processed Traffic Data for ML Algorithms/" {DATA_LOCAL} --recursive
    return DATA_LOCAL

def ensure_parquet():
    if os.path.exists(f'{WORK}/parquet/meta.json'):
        return True
    if os.path.exists(f'{PARQUET_BACKUP}/meta.json'):
        !rm -rf {WORK}/parquet && cp -r "{PARQUET_BACKUP}" {WORK}/parquet
        return True
    return False

print('parquet:', ensure_parquet())
print({d: n_csv(d) for d in (DATA_DRIVE, f'{PROJECT}/data', DATA_LOCAL)})

## 1. ETL và báo cáo chất lượng dữ liệu

Đọc 10 file CSV (~16 triệu luồng), làm sạch, khử trùng lặp, ghi Parquet. Parquet được sao lưu lên Drive, nên ô này chỉ cần chạy **một lần**.

In [ ]:
%%time
if not ensure_parquet():
    DATA = data_dir()
    print(n_csv(DATA), 'CSV in', DATA)
    !python -m ids.etl --input "{DATA}" --output {WORK}/parquet --report "{RESULTS}/quality_report.json"
    assert os.path.exists(f'{WORK}/parquet/meta.json'), 'ETL failed'
    !rm -rf "{PARQUET_BACKUP}" && cp -r {WORK}/parquet "{PARQUET_BACKUP}"
!du -sh {WORK}/parquet/clean {WORK}/parquet/dedup

## 2. Kịch bản 1: chia ngẫu nhiên, có trùng lặp và đã khử trùng lặp

Huấn luyện LR, DT, RF, GBT trên hai biến thể dữ liệu; quét ngưỡng chi phí trên validation; chọn mô hình có chi phí nhỏ nhất ở tỷ lệ 1:50 và lưu vào `colab_out/models`.

In [ ]:
%%time
assert ensure_parquet()
!python -m ids.train scenario1 --parquet {WORK}/parquet --work {WORK} --results "{RESULTS}" --models-out "{MODELS}" --preset full --train-frac {S1_TRAIN_FRAC}

## 3. Random Forest đa lớp (7 họ tấn công)

In [ ]:
%%time
assert ensure_parquet()
!python -m ids.train multiclass --parquet {WORK}/parquet --work {WORK} --results "{RESULTS}" --models-out "{MODELS}" --preset full --train-frac {S1_TRAIN_FRAC}

## 4. Kịch bản 2: leave-one-day-out

40 lần huấn luyện (10 ngày × 4 mô hình). Kết quả được ghi sau mỗi ngày; nếu bị ngắt phiên, chạy lại ô này và các ngày đã xong sẽ được bỏ qua (`--resume`).

In [ ]:
%%time
assert ensure_parquet()
!python -m ids.train scenario2 --parquet {WORK}/parquet --work {WORK} --results "{RESULTS}" --models-out "{MODELS}" --preset {S2_PRESET} --train-frac {S2_TRAIN_FRAC} --resume

## 5. Benchmark khả năng mở rộng: Spark và pandas/scikit-learn

Mẫu dữ liệu thật 25% ⊂ 50% ⊂ 100%, số core 1, 2, 4, 8 (tới số vCPU của máy). Mỗi lần đo chạy trong tiến trình riêng, ghi thời gian, RAM đỉnh và trạng thái (`ok`, `oom`, `timeout`). Kết quả đo được ghi dần nên có thể chạy tiếp bằng `--resume`.

In [ ]:
%%time
assert ensure_parquet()
DATA = data_dir()
!python -m ids.bench prepare --data "{DATA}" --parquet {WORK}/parquet --bench {WORK}/bench
!python -m ids.bench all --data "{DATA}" --parquet {WORK}/parquet --bench {WORK}/bench --cores {BENCH_CORES} --fracs {BENCH_FRACS} --timeout {BENCH_TIMEOUT} --spark-mem {BENCH_MEM} --note "Google Colab {N_CPU} vCPU {RAM_GB} GB" --out "{RESULTS}/scalability.json" --resume

## 6. Biểu đồ và bảng tổng hợp

In [ ]:
!python -m ids.figures --results "{RESULTS}"

from IPython.display import Image, Markdown, display
display(Markdown(open(f'{RESULTS}/summary.md', encoding='utf-8').read()))
for f in sorted(os.listdir(f'{RESULTS}/figures')):
    print(f)
    display(Image(f'{RESULTS}/figures/{f}'))

## 7. Kiểm tra và bàn giao kết quả

1. Tải thư mục `MyDrive/ie212-ids2018/colab_out/` về máy, đặt vào `colab_out/` trong repo.
2. Ghi lại dòng cấu hình in ở ô thiết lập và `Wall time` của từng bước.
3. Muốn chạy demo bằng mô hình mới: chép `colab_out/models/` đè lên `models/`.
4. Tạo nhánh mới, commit `colab_out/` (không commit Parquet), mở Pull Request.

In [ ]:
!du -sh "{RESULTS}" "{MODELS}"
!ls -la "{RESULTS}" "{MODELS}"